In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,r2_score

In [ ]:
data = fetch_california_housing()

X = data.data
y = data.target

In [ ]:
df = pd.DataFrame(data.data,columns=data.feature_names)

In [ ]:
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [ ]:
df.isnull().sum()

,0
MedInc,0
HouseAge,0
AveRooms,0
AveBedrms,0
Population,0
AveOccup,0
Latitude,0
Longitude,0


In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=0)

In [ ]:
scale = StandardScaler()

X_train_scale = scale.fit_transform(X_train)
X_test_scale = scale.transform(X_test)

In [ ]:
X_train_tens = torch.tensor(X_train_scale,dtype=torch.float32)
X_test_tens = torch.tensor(X_test_scale,dtype=torch.float32)

y_train_tens = torch.tensor(y_train,dtype=torch.float32).view(-1,1)
y_test_tens = torch.tensor(y_test,dtype=torch.float32).view(-1,1)

In [ ]:
class ANNRegression(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(8,64)
    self.fc2 = nn.Linear(64,32)
    self.fc3 = nn.Linear(32,16)
    self.fc4 = nn.Linear(16,8)
    self.fc5 = nn.Linear(8,4)
    self.output = nn.Linear(4,1)

    self.relu = nn.ReLU()

  def forward(self,x):
    x = self.relu(self.fc1(x))
    x = self.relu(self.fc2(x))
    x = self.relu(self.fc3(x))
    x = self.relu(self.fc4(x))
    x = self.relu(self.fc5(x))

    x = self.output(x)

    return x

In [ ]:
model = ANNRegression()

criterion = nn.MSELoss()

optimizer = optim.Adam(model.parameters(),lr=0.001)

In [ ]:
epochs = 1000
for epoch in range(epochs):
  # Forward
  y_pred = model(X_train_tens)

  # Loss
  loss = criterion(y_pred,y_train_tens)

  # clear old gradients
  optimizer.zero_grad()

  # backpropagation
  loss.backward()

  # update weights
  optimizer.step()

  if epoch % 20 == 0:
    print(f"Epoch: {epoch}, Loss: {loss.item()}")

Epoch: 0, Loss: 5.688075542449951
Epoch: 20, Loss: 5.250163555145264
Epoch: 40, Loss: 4.49156379699707
Epoch: 60, Loss: 1.766793131828308
Epoch: 80, Loss: 1.051729679107666
Epoch: 100, Loss: 0.8442673087120056
Epoch: 120, Loss: 0.7604430913925171
Epoch: 140, Loss: 0.7053220272064209
Epoch: 160, Loss: 0.6518369317054749
Epoch: 180, Loss: 0.5920220017433167
Epoch: 200, Loss: 0.526326060295105
Epoch: 220, Loss: 0.4636540710926056
Epoch: 240, Loss: 0.42400234937667847
Epoch: 260, Loss: 0.40044963359832764
Epoch: 280, Loss: 0.38434967398643494
Epoch: 300, Loss: 0.37275514006614685
Epoch: 320, Loss: 0.3636836111545563
Epoch: 340, Loss: 0.3555384874343872
Epoch: 360, Loss: 0.3483005166053772
Epoch: 380, Loss: 0.3410891890525818
Epoch: 400, Loss: 0.3334398567676544
Epoch: 420, Loss: 0.3254643380641937
Epoch: 440, Loss: 0.3170296251773834
Epoch: 460, Loss: 0.3093395233154297
Epoch: 480, Loss: 0.30339315533638
Epoch: 500, Loss: 0.2989930808544159
Epoch: 520, Loss: 0.29493558406829834
Epoch: 540,

In [ ]:
with torch.no_grad():
  train_prediction = model(X_train_tens)
  test_prediction = model(X_test_tens)

  train_mse = mean_squared_error(y_train_tens,train_prediction)
  train_r2 = r2_score(y_train_tens,train_prediction)

  test_mse = mean_squared_error(y_test_tens,test_prediction)
  test_r2 = r2_score(y_test_tens,test_prediction)

  print(f"Training MSE: {train_mse}")
  print(f"Training R2 Score: {train_r2:.2%}")

  print(f"Testing MSE: {test_mse}")
  print(f"Testing R2 Score: {test_r2:.2%}")

Training MSE: 0.2550384914074837
Training R2 Score: 80.94%
Testing MSE: 0.28532185514970526
Testing R2 Score: 78.12%


In [ ]:
new_house = np.array([
    [5.0,20,6,1,1000,3,34,-118]
])

In [ ]:
new_house_scaled = scale.transform(new_house)

In [ ]:
new_house_tens = torch.tensor(new_house_scaled,dtype=torch.float32)

In [ ]:
with torch.no_grad():
  prediction = model(new_house_tens)
  print("House Price: ",prediction.item())

House Price:  2.505800724029541
